# Clean the Residential Care Demand by Service LGA table from Excel

This notebook builds a simple data pipeline for the uploaded Excel file and only reads the required worksheet.
It cleans the residential care table and reshapes the admission type columns into a long-format table for loading into Postgres.


In [ ]:
from pathlib import Path
import pandas as pd

INPUT_XLSX = Path("../input/gen_data_residential_care_demand_by_service_loc.xlsx")
SHEET_NAME = "Table 4.1 (LGA)"

INPUT_XLSX, SHEET_NAME

(PosixPath('input/gen_data_residential_care_demand_by_service_loc.xlsx'),
 'Table 4.1 (LGA)')

In [2]:
raw_df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME,
    header=None
)

raw_df.head(10)

,0,1,2,3,4
0,Table 4.1: Number of people using residential ...,NaN,NaN,NaN,NaN
1,Return to contents,NaN,NaN,NaN,NaN
2,LGA code,LGA name,Permanent,Respite,Total
3,10050,Albury,609,24,633
4,10180,Armidale,221,10,231
5,10250,Ballina,621,4,625
6,10300,Balranald,8,4,12
7,10470,Bathurst,478,15,493
8,10500,Bayside (NSW),1221,64,1285
9,10550,Bega Valley,306,9,315


In [3]:
def clean_residential_care_demand(input_path: Path, sheet_name: str = "Table 4.1 (LGA)"):
    # Read only the required worksheet without assuming the header row position.
    df = pd.read_excel(input_path, sheet_name=sheet_name, header=None)

    # Find the real header row by matching the first column name.
    header_mask = (
        df.iloc[:, 0].astype("string").str.contains(r"^LGA code$", na=False)
    )

    if not header_mask.any():
        raise ValueError("Could not find the header row for the LGA table.")

    header_row_idx = header_mask[header_mask].index[0]

    # Rebuild the dataframe using the detected header row.
    df = df.iloc[header_row_idx + 1:].copy()
    df.columns = df.iloc[0:0].reindex(columns=df.columns).columns  # keep column count stable
    df.columns = pd.Index(pd.read_excel(input_path, sheet_name=sheet_name, header=None).iloc[header_row_idx].tolist())

    # Keep only the required columns from the worksheet.
    keep_columns = ["LGA code", "LGA name", "Permanent", "Respite"]
    df = df[keep_columns].copy()

    # Drop completely empty rows.
    df = df.dropna(axis=0, how="all")

    # Rename columns.
    df = df.rename(columns={
        "LGA code": "lga_code",
        "LGA name": "lga_name",
        "Permanent": "permanent",
        "Respite": "respite",
    })

    # Clean text columns: normalize whitespace and convert blanks to missing values.
    text_cols = ["lga_code", "lga_name", "permanent", "respite"]
    for col in text_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    # Remove repeated header rows if they appear in the body.
    df = df[df["lga_code"] != "LGA code"].copy()

    # Treat the code as strings.
    df["lga_code"] = (
        pd.to_numeric(df["lga_code"], errors="coerce")
        .astype("Int64").astype("string")
    )

    # Clean numeric columns.
    numeric_cols = ["permanent", "respite"]
    for col in numeric_cols:
        df[col] = (
            df[col]
            .str.replace(",", "", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
            .astype("Int64")
        )

    # Final row filtering: keep rows with a valid LGA code and name.
    df = df[df["lga_code"].notna()].copy()
    df = df[df["lga_name"].notna()].copy()

    # Remove exact duplicates in wide form first.
    rows_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = rows_before - len(df)

    # Reshape to long format for ML-ready / normalized loading.
    long_df = df.melt(
        id_vars=["lga_code", "lga_name"],
        value_vars=["permanent", "respite"],
        var_name="admission_type",
        value_name="people_count",
    )

    # Clean reshaped values.
    long_df["admission_type"] = long_df["admission_type"].replace({
        "permanent": "Permanent",
        "respite": "Respite",
    })

    long_df["people_count"] = pd.to_numeric(long_df["people_count"], errors="coerce").astype("Int64")

    # Keep rows with valid keys and counts.
    long_df = long_df[long_df["lga_code"].notna()].copy()
    long_df = long_df[long_df["lga_name"].notna()].copy()
    long_df = long_df[long_df["admission_type"].notna()].copy()
    long_df = long_df[long_df["people_count"].notna()].copy()

    long_df = long_df.drop_duplicates().reset_index(drop=True)

    # Return a summary of the cleaning process.
    summary = {
        "sheet_name": sheet_name,
        "wide_output_rows": len(df),
        "long_output_rows": len(long_df),
        "duplicates_removed_wide": duplicates_removed,
        "column_count_long": long_df.shape[1],
        "null_lga_code_rows": int(long_df["lga_code"].isna().sum()),
        "null_lga_name_rows": int(long_df["lga_name"].isna().sum()),
        "null_people_count_rows": int(long_df["people_count"].isna().sum()),
    }

    return long_df, summary

In [4]:
clean_df, summary = clean_residential_care_demand(INPUT_XLSX, SHEET_NAME)
summary

{'sheet_name': 'Table 4.1 (LGA)',
 'wide_output_rows': 387,
 'long_output_rows': 774,
 'duplicates_removed_wide': 0,
 'column_count_long': 4,
 'null_lga_code_rows': 0,
 'null_lga_name_rows': 0,
 'null_people_count_rows': 0}

In [5]:
clean_df.head()

,lga_code,lga_name,admission_type,people_count
0,10050,Albury,Permanent,609
1,10180,Armidale,Permanent,221
2,10250,Ballina,Permanent,621
3,10300,Balranald,Permanent,8
4,10470,Bathurst,Permanent,478


In [6]:
print(f"Blank rows remaining: {int(clean_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining: {int(clean_df.duplicated().sum())}")
print(f"Null lga_code rows: {int(clean_df['lga_code'].isna().sum())}")
print(f"Null lga_name rows: {int(clean_df['lga_name'].isna().sum())}")
print(f"Data types:\n{clean_df.dtypes}")

Blank rows remaining: 0
Exact duplicates remaining: 0
Null lga_code rows: 0
Null lga_name rows: 0
Data types:
lga_code          string
lga_name          string
admission_type       str
people_count       Int64
dtype: object


# Insert Clean Data into DB

In [7]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/residential_care_demand_by_service_lga.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [8]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_df.copy()

target_columns = [
    "lga_code",
    "lga_name",
    "admission_type",
    "people_count",
]

df_insert = df_insert[target_columns].copy()

text_cols = [
    "lga_code",
    "lga_name",
    "admission_type",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

int_cols = [
    "people_count",
]

for col in int_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce").astype("Int64")

df_insert.head()

,lga_code,lga_name,admission_type,people_count
0,10050,Albury,Permanent,609
1,10180,Armidale,Permanent,221
2,10250,Ballina,Permanent,621
3,10300,Balranald,Permanent,8
4,10470,Bathurst,Permanent,478


In [9]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "residential_care_demand_by_service_lga",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into residential_care_demand_by_service_lga")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM residential_care_demand_by_service_lga"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 774 rows into residential_care_demand_by_service_lga
Rows in table: 774
